In [1]:

!export HF_HOME=/opt/wisper/models
!export HUGGINGFACE_HUB_CACHE=/opt/wisper/models
!export TRANSFORMERS_CACHE=/opt/wisper/models

# ОБЯЗАТЕЛЬНО
!export HF_TOKEN=hf_Hdfgdgfdsfgdfgdgdsgfsdfdfgdfgdfg

# В runtime
!export HF_HUB_OFFLINE=1
!export TRANSFORMERS_OFFLINE=1
!export use_auth_token="hf_Hdfgdgfdsfgdfgdgdsgfsdfdfgdfgdfg"


In [2]:
# local_diarization.py
import os
import torch
from pyannote.audio import Pipeline
from whisperx.diarize import DiarizationPipeline
from whisperx.log_utils import get_logger

logger = get_logger(__name__)


class LocalDiarizationPipeline(DiarizationPipeline):
    def __init__(
        self,
        device="cpu",
        cache_dir="./models",
    ):
        os.environ.setdefault("HF_HUB_OFFLINE", "1")
        os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

        logger.info("Loading diarization pipeline (offline, local cache)")

        self.model = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1",
            cache_dir=cache_dir,
            #use_auth_token=True,  # берёт HF_TOKEN
        ).to(torch.device(device))


/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


In [5]:
import whisperx
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Whisper
model = whisperx.load_model(
    "large-v3",
    device=device,
    compute_type="int8",
    download_root="./models",
)

audio = whisperx.load_audio("audio.wav")
result = model.transcribe(audio)


/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


2025-12-13 20:33:34 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2025-12-13 20:33:34 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint .venv/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.4.0. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.8.0+cu128. Bad things might happen unless you revert torch to 1.x.
2025-12-13 20:33:43 - whisperx.asr - INFO - Detected language: ru (0.99) in first 30s of audio


In [6]:

diarize = LocalDiarizationPipeline(device=device)

2025-12-13 20:34:36 - whisperx - INFO - Loading diarization pipeline (offline, local cache)


/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


In [7]:

diarize_df = diarize(audio)

# Привязка слов к спикерам
result = whisperx.assign_word_speakers(
    diarize_df,
    result,
)


/home/slava/Documents/projects/Hanzo/wisper_python31013/.venv/lib/python3.10/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


In [8]:
result

{'segments': [{'text': ' Да. Доброе утро. Доброе утро. А время сколько-то сейчас? Шесть часов. Твою мать, я только уснула. А что ты делала-то? Что-то что-то. То на день, то так. Крутилась-крутилась. Никак. Вот так вот. Ей-то еще где была. В три часа ночи звонит. Ты доехала до дома?',
   'start': 1.55,
   'end': 31.182,
   'speaker': 'SPEAKER_00'},
  {'text': ' Я говорю, я уже давно доехала. Ну и все. И сон закончился. Что дома сидишь, приезжай в гости. Ну давай. Ну вот до трех я кое-как заснул. Как бы домой сейчас не вернулась.',
   'start': 31.655,
   'end': 57.49,
   'speaker': 'SPEAKER_00'},
  {'text': ' Ясно? Почему? Алло. Да, я слышал. Вот так вот. И такой вариант возможен. Хорошо услышал телефон, что звенит. Ага. Как самочувствие ваше? Ясно.',
   'start': 63.245,
   'end': 93.13,
   'speaker': 'SPEAKER_00'},
  {'text': ' Ясно. Ну ладно, пошла я вставать потихоньку. Давай. Позвонишь. Если на работе, то не возьму трубку. Если дома, значит, возьму. Давай.',
   'start': 93.282,
   'e